In [1]:
import numpy as np
from jax import numpy as jnp

In [2]:
class FlowValue:
    def __init__(self, name: str, val: float):
        self.name = name
        self.val = val

    def __repr__(self):
        return f"FlowValue: {self.name}({self.val})"

class AdjustmentArray:
    def __init__(self, name, arr):
        self.name = name
        self.arr = arr

    def __repr__(self):
        return f"AdjArray: {self.name}"

class ArrayAdjusterValue:
    def __init__(self, group, idx):
        self.group = group
        self.idx = idx

    def __repr__(self):
        return f"ArrayAdj: {self.group.name}[{self.idx}]"






In [3]:
src_comps = np.array([0,1,2,3])
dest_comps = np.array([1,2,3,4])

comp_vals = np.array([1.0,1.2,1.4,1.6,1.8])

In [ ]:
class FlowTable:
    def __init__(self, src_comps, base_value):
        self.src_comps = src_comps
        self.adj_table = [[base]

        base_val = FlowValue("base", )

cval_adj_array = AdjustmentArray("comp_vals", comp_vals)

flow_adj_table = [[base_val] for i in range(4)]
flow_adj_table = [f + [ArrayAdjusterValue(cval_adj_array, src_comps[i])] for i,f in enumerate(flow_adj_table)]
flow_adj_table


In [2]:
import computegraph as cg

In [3]:
from typing import NamedTuple
from jax_dataclasses import pytree_dataclass

In [8]:
from dataclasses import field
import dataclasses
from copy import deepcopy

In [10]:
def default(value) -> dataclasses.Field:
    if isinstance(value, float):
        return field(default = value) # type: ignore
    else:
        return field(default_factory=lambda: deepcopy(value))

In [11]:
import jax

In [12]:
@pytree_dataclass
class JStrainParams:
    rel_inf: float = default(1.0)

@pytree_dataclass
class JPopParams:
    population: float = 1000.0

@pytree_dataclass
class JRandomProcessParams:
    proc_vals: jax.Array = default(jnp.zeros(100))

@pytree_dataclass
class JModelParams:
    contact_rate: float = 1.0
    strains: dict[str, JStrainParams] = default({"s0": JStrainParams()})
    pop: JPopParams = JPopParams()
    random_process: JRandomProcessParams = JRandomProcessParams()


In [16]:
from dataclasses import dataclass


In [17]:
from types import GenericAlias

In [315]:
class Parameter:
    def __init__(self, name, default, base_path=None):
        self.name = name
        self.default = default
        self.base_path = base_path or []

    def __repr__(self):
        return f"Parameter {".".join(self.base_path +[self.name])}({self.default})"

In [316]:
Parameter("x", 1.0, ["strain","0"])

Parameter strain.0.x(1.0)

In [317]:
field = JModelParams.__dataclass_fields__["strain"]

def is_dict_field(field):
    ftype = field.type

    if isinstance(ftype, GenericAlias):
        return ftype.__origin__ == dict
    

def get_dict_proxy(field, base_path):
    ftype = field.type
    ktype, vtype = ftype.__args__

    class DictProxy:
        def __init__(self, vtype, base_path):
            self.vtype = vtype
            self.base_path = base_path

        def __getitem__(self, k) -> :
            return make_pt_proxy(self.vtype, self.base_path + [k])
        
    return DictProxy(vtype, base_path)
        
dp = get_dict_proxy(field, ["strain"])

dp["shoes"]

SyntaxError: expected ':' (1798369049.py, line 19)

In [318]:
def get_default(field):
    if not isinstance(field.default, _MISSING_TYPE):
        return field.default
    elif not isinstance(field.default_factory, _MISSING_TYPE):
        return field.default_factory()
    else:
        return None
    

def make_pt_proxy(mp_class: type, name_ext=None):
    name_ext = name_ext or []
    param_fields = mp_class.__dataclass_fields__

    out_k = []
    out_v = []
    for k, field in param_fields.items():
        kj = ".".join(name_ext+[k])
        if hasattr(field.type, "__dataclass_fields__"):
            out_val = make_pt_proxy(field.type, name_ext+[k])
        elif is_dict_field(field):
            out_val = get_dict_proxy(field, name_ext+[k])
        else:
            out_val = Parameter(k, get_default(field), name_ext)
        #else:
        #    raise TypeError("Unsupported parameter type", k, field)
        out_k.append(k)
        out_v.append(out_val)
    typename = f"{mp_class.__name__}Proxy"

    proxy_type = NamedTuple(typename, [(k,type(v)) for k,v in zip(out_k, out_v)]) #type: ignore
    return proxy_type(*out_v)


In [330]:
strains = {
    "s0": JStrainParams(1.5),
    "s1": JStrainParams(1.2)
}
jprox = make_pt_proxy(JModelParams)

In [331]:
lookups = []
for s in strains:
    strain_params = jprox.strains[s]
    lookups.append(strain_params.rel_inf)

actual_params = JModelParams(strains=strains)

In [332]:
def lookup_param(p: Parameter, mclass):
    source = mclass
    for group in p.base_path:
        if isinstance(source, dict):
            source = source[group]
        else:
            source = getattr(source, group)
    return getattr(source, p.name)

In [333]:
@jit
def prod_lookups(params):
    return jnp.array([lookup_param(l, params) for l in lookups])

In [334]:
prod_lookups(actual_params)

Array([1.5, 1.2], dtype=float32)

In [325]:
lookup_param(lookups[0],actual_params)

1.0

In [155]:
from dataclasses import _MISSING_TYPE

In [157]:
JModelParams.__dataclass_fields__["strain"].default_factory()

{'s0': JStrainParams(rel_inf=1.0)}

In [127]:
j0 = JModelParams(pop=JPopParams(522.0))
j1 = JModelParams(pop=JPopParams(522.0))

j0.strain["s0"] == j1.strain["s0"]

True

In [345]:
def nested(params):
    return params.strains["s0"].rel_inf ** 2.0

def nested2(params):
    return params.rel_inf ** 2.0


@jit
def thing(params):
    return params.pop.population * nested(params)

@jit
def thing2(params):
    return params.pop.population * params.strains["s0"].rel_inf ** 2.0


@jit
def thing3(params):
    return params.pop.population * nested2(params.strains["s0"])


thing(actual_params)

jax.make_jaxpr(thing3)(actual_params)

{ lambda ; a:f32[] b:f32[] c:f32[] d:f32[] e:f32[100]. let
    f:f32[] = pjit[
      name=thing3
      jaxpr={ lambda ; g:f32[] h:f32[] i:f32[] j:f32[] k:f32[100]. let
          l:f32[] = pow h 2.0
          m:f32[] = mul j l
        in (m,) }
    ] a b c d e
  in (f,) }

In [129]:
def strain_pops(params: JModelParams):
    return {k: strain.rel_inf * params.pop.population for k, strain in params.strain.items()}

In [132]:
strain_pops(JModelParams(pop=JPopParams(2000.0),strain={"s0": JStrainParams(1.0), "s1": JStrainParams(1.5)}))

{'s0': 2000.0, 's1': 3000.0}

In [133]:
from jax import jit
import jax

@jit
def new_def_pop(p):
    return JModelParams(pop=JPopParams(p)).pop.population

@jit
def retp(p):
    return p

In [116]:
jax.make_jaxpr(retp)(5.1)

{ lambda ; a:f32[]. let
    pjit[name=retp jaxpr={ lambda ; b:f32[]. let  in () }] a
  in (a,) }

In [115]:
jax.make_jaxpr(new_def_pop)(5.1)

{ lambda ; a:f32[]. let
    pjit[name=new_def_pop jaxpr={ lambda ; b:f32[]. let  in () }] a
  in (a,) }

In [109]:
new_def_pop(5.0)

JModelParams(contact_rate=Array(1., dtype=float32, weak_type=True), strain={'s0': JStrainParams(rel_inf=Array(1., dtype=float32, weak_type=True))}, pop=JPopParams(population=Array(5., dtype=float32, weak_type=True)))

In [34]:
ModelParams._fields

('contact_rate', 'strain', 'pop')

In [35]:
ModelParams.__annotations__

{'contact_rate': float,
 'strain': __main__.StrainParams,
 'pop': __main__.PopParams}

In [45]:
def is_namedtupleclass(obj):
    return (tuple in obj.__bases__) and hasattr(obj, "_fields")

In [49]:
ModelParams.__name__

'ModelParams'

In [77]:
def make_mp_proxy(mp_class: type, name_ext=None):
    name_ext = name_ext or []
    param_fields = mp_class.__annotations__

    out_k = []
    out_v = []
    for k, v in param_fields.items():
        kj = ".".join(name_ext+[k])
        if is_namedtupleclass(v):
            out_val = make_mp_proxy(v, name_ext+[k])
        elif v == float:
            out_val = Parameter(kj)
        else:
            raise TypeError("Unsupported parameter type", k, v)
        out_k.append(k)
        out_v.append(out_val)
    typename = f"{mp_class.__name__}Proxy"

    proxy_type = NamedTuple(typename, [(k,type(v)) for k,v in zip(out_k, out_v)]) #type: ignore
    return proxy_type(*out_v)


In [78]:
pp = make_mp_proxy(ModelParams)

In [79]:
pp.contact_rate

In [73]:
pt(*pv)

ModelParamsProxy(contact_rate=<__main__.Parameter object at 0x000001F2FA14DB50>, strain=StrainParamsProxy(rel_inf=[<__main__.Parameter object at 0x000001F2FA1590F0>]), pop=PopParamsProxy(population=[<__main__.Parameter object at 0x000001F2FA158C20>]))

In [50]:
a = [0,1]
b = ["a", "b"]

[(ai, bi) for ai, bi in zip(a,b)]

[(0, 'a'), (1, 'b')]

In [25]:
import jax

In [26]:
class Parameter:
    def __init__(self,name):
        self.name = name

In [27]:
leaves, treedef = jax.tree_flatten(ModelParams(Parameter("x"),Parameter("y"), {"s0": Parameter("s0")}))

In [28]:
treedef

PyTreeDef(CustomNode(namedtuple[ModelParams], [*, *, {'s0': *}]))

In [20]:
jax.tree_unflatten(treedef,leaves)

ModelParams(x=<__main__.Parameter object at 0x000001F2F8EFF820>, y=<__main__.Parameter object at 0x000001F2F8EFF490>, thing={'s0': <__main__.Parameter object at 0x000001F2F8F83E30>})

In [7]:
ModelParams.__annotations__

{'x': float, 'y': float}

In [28]:
params = ModelParams(2.1,1.1)

In [17]:
base_val = FlowValue("base", 0.1)

cval_adj_array = AdjustmentArray("comp_vals", comp_vals)

flow_adj_table = [[base_val] for i in range(4)]
flow_adj_table = [f + [ArrayAdjusterValue(cval_adj_array, src_comps[i])] for i,f in enumerate(flow_adj_table)]
flow_adj_table


[[FlowValue: base(0.1), ArrayAdj: comp_vals[0]],
 [FlowValue: base(0.1), ArrayAdj: comp_vals[1]],
 [FlowValue: base(0.1), ArrayAdj: comp_vals[2]],
 [FlowValue: base(0.1), ArrayAdj: comp_vals[3]]]